In [1]:
import os
import pandas as pd
import shutil
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Import global paths from config.py
import config


Data path: /Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/data
Raw images path: /Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/data/raw/images
Labels path: /Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/data/Train.csv
CompTest path: /Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/data/Test.csv
Results path: /Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/results
Processed images path: /Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/data/processed
Processed labels path: /Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/data/augmented_labels.csv


In [2]:
# Use paths from config.py
data_path = config.GLOBAL_CONFIG_DATA_PATH
raw_images_path = config.GLOBAL_CONFIG_DATA_RAW_IMAGE_PATH
annotations_path = config.GLOBAL_CONFIG_DATA_RAW_LABEL_PATH  # This should point to 'Train.csv'

# Define the edgeData directory
edge_data_dir = os.path.join(data_path, 'edgeData')
os.makedirs(edge_data_dir, exist_ok=True)


In [3]:
# Directories for images and labels within edgeData
images_dir = os.path.join(edge_data_dir, 'images')
labels_dir = os.path.join(edge_data_dir, 'labels')
os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Create train and val directories
train_images_dir = os.path.join(images_dir, 'train')
val_images_dir = os.path.join(images_dir, 'val')
train_labels_dir = os.path.join(labels_dir, 'train')
val_labels_dir = os.path.join(labels_dir, 'val')

os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(val_images_dir, exist_ok=True)
os.makedirs(train_labels_dir, exist_ok=True)
os.makedirs(val_labels_dir, exist_ok=True)


In [4]:
# Load bounding box data from the CSV file
annotations = pd.read_csv(annotations_path)

# Map classes to indices
class_mapping = {'Trophozoite': 0, 'WBC': 1, 'NEG': 2}


In [5]:
# Get unique images from annotations
unique_images = annotations['Image_ID'].unique()

# Split into training and validation sets
train_images, val_images = train_test_split(unique_images, test_size=0.2, random_state=42)


In [6]:
def process_and_copy_images(image_ids, split):
    if split == 'train':
        images_dest_dir = train_images_dir
        labels_dest_dir = train_labels_dir
    else:
        images_dest_dir = val_images_dir
        labels_dest_dir = val_labels_dir

    # Log file to track processed images
    processed_log_path = os.path.join(edge_data_dir, f'processed_images_{split}.log')
    if not os.path.exists(processed_log_path):
        with open(processed_log_path, 'w') as file:
            file.write("")  # Create an empty log file

    # Load processed images from the log file
    with open(processed_log_path, 'r') as file:
        processed_images = file.read().splitlines()

    for image_id in tqdm(image_ids, desc=f"Processing {split} images"):
        if image_id not in processed_images:
            src_image_path = os.path.join(raw_images_path, image_id)
            dst_image_path = os.path.join(images_dest_dir, image_id)

            # Copy image if it doesn't exist in destination
            if not os.path.exists(dst_image_path):
                if os.path.exists(src_image_path):
                    shutil.copy(src_image_path, dst_image_path)
                else:
                    print(f"File not found: {src_image_path}")
                    continue

            # Read image to get dimensions
            try:
                image = cv2.imread(dst_image_path)
                if image is None:
                    raise ValueError(f"Unable to read image at {dst_image_path}")
                height, width, _ = image.shape
            except Exception as e:
                print(f"Error processing image {dst_image_path}: {e}")
                continue

            label_file_path = os.path.join(labels_dest_dir, os.path.splitext(image_id)[0] + '.txt')

            # Process and write labels only if they have not been created
            if not os.path.exists(label_file_path):
                image_annotations = annotations[annotations['Image_ID'] == image_id]
                with open(label_file_path, 'w') as file:
                    for _, row in image_annotations.iterrows():
                        class_name = row['class']
                        if class_name == 'NEG' and row['xmin'] == 0 and row['xmax'] == 0:
                            continue  # Skip negative samples with no bounding box
                        class_id = class_mapping[class_name]
                        xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']

                        # Convert to YOLO format
                        x_center = ((xmin + xmax) / 2) / width
                        y_center = ((ymin + ymax) / 2) / height
                        bbox_width = (xmax - xmin) / width
                        bbox_height = (ymax - ymin) / height

                        # Write to label file
                        file.write(f"{class_id} {x_center} {y_center} {bbox_width} {bbox_height}\n")

            # Log this image as processed
            with open(processed_log_path, 'a') as log_file:
                log_file.write(f"{image_id}\n")


In [7]:
process_and_copy_images(train_images, 'train')
process_and_copy_images(val_images, 'val')


Processing val images: 100%|██████████████████| 550/550 [10:11<00:00,  1.11s/it]


In [15]:
# Paths in data.yaml should be relative to the data.yaml file
data_yaml_content = f"""
train: {os.path.relpath(train_images_dir, edge_data_dir)}
val: {os.path.relpath(val_images_dir, edge_data_dir)}

nc: {len(class_mapping)}
names: {list(class_mapping.keys())}
"""

data_yaml_path = os.path.join(edge_data_dir, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)


In [16]:
# Clone YOLOv5 repository
!git clone https://github.com/ultralytics/yolov5
%cd yolov5

# Install dependencies
%pip install -qr requirements.txt


Cloning into 'yolov5'...
remote: Enumerating objects: 17022, done.
remote: Total 17022 (delta 0), reused 0 (delta 0), pack-reused 17022 (from 1)
Receiving objects: 100% (17022/17022), 15.61 MiB | 4.91 MiB/s, done.
Resolving deltas: 100% (11690/11690), done.
/Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/notebooks/yolov5/yolov5/yolov5
Note: you may need to restart the kernel to use updated packages.


In [18]:
import os
print(os.getcwd())


/Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/notebooks/yolov5/yolov5/yolov5


In [19]:
import os
desired_path = '/Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/notebooks/yolov5'
os.chdir(desired_path)
print("Current Working Directory: ", os.getcwd())


Current Working Directory:  /Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/notebooks/yolov5


In [ ]:
# Start training
!python train.py --img 640 --batch 16 --epochs 50 --data "{data_yaml_path}" --weights yolov5s.pt
# Run inference on validation set
best_weights = 'runs/train/exp/weights/best.pt'  # Adjust if necessary
!python detect.py --weights "{best_weights}" --source "{val_images_dir}" --save-txt --save-conf


train: weights=yolov5s.pt, cfg=, data=/Users/frankoswanepoel/Desktop/2024/COS 711/Assingments/Assingment3/COS711_Assingment3/data/edgeData/data.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=50, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=8, project=runs/train, name=exp, exist_ok=False, quad=False, cos_lr=False, label_smoothing=0.0, patience=100, freeze=[0], save_period=-1, seed=0, local_rank=-1, entity=None, upload_dataset=False, bbox_interval=-1, artifact_alias=latest, ndjson_console=False, ndjson_file=False
github: up to date with https://github.com/ultralytics/yolov5 ✅
fatal: cannot change to '/Users/frankoswanepoel/Desktop/2024/COS': No such file or directory
YOLOv5 🚀 2024-11-3 Python-3.11.5 torch-2.2.2 CPU

hyperparameters: 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import glob

def plot_image_with_boxes(image_path, boxes, YOLO_FORMAT=True, show_labels=False):
    try:
        image = cv2.imread(image_path)
        if image is None:
            print(f"Failed to load image at {image_path}")
            return
        h, w, _ = image.shape
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(10, 10))
        plt.imshow(image)

        for box in boxes:
            class_id, x_center, y_center, bbox_width, bbox_height = box
            if YOLO_FORMAT:
                xmin = int((x_center - bbox_width / 2) * w)
                ymin = int((y_center - bbox_height / 2) * h)
                xmax = int((x_center + bbox_width / 2) * w)
                ymax = int((y_center + bbox_height / 2) * h)
            else:
                xmin, ymin, xmax, ymax = box[1:5]

            rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                     linewidth=2, edgecolor='red', facecolor='none')
            plt.gca().add_patch(rect)
            if show_labels:
                plt.text(xmin, ymin - 10, f'Class {int(class_id)}', color='red', fontsize=12, weight='bold')

        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"An error occurred: {e}")

# Path to predictions
predictions_path = 'runs/detect/exp/labels'  # Adjust if necessary

for label_file in glob.glob(os.path.join(predictions_path, '*.txt')):
    image_id = os.path.basename(label_file).replace('.txt', '.jpg')
    image_path = os.path.join(val_images_dir, image_id)

    # Read predicted bounding boxes
    boxes = []
    with open(label_file, 'r') as f:
        for line in f:
            elements = line.strip().split()
            class_id = int(elements[0])
            x_center = float(elements[1])
            y_center = float(elements[2])
            bbox_width = float(elements[3])
            bbox_height = float(elements[4])
            boxes.append([class_id, x_center, y_center, bbox_width, bbox_height])

    # Plot the image with bounding boxes
    plot_image_with_boxes(image_path, boxes, YOLO_FORMAT=True, show_labels=True)
